# TALON vs EAGLE-3 Benchmark

This notebook benchmarks TALON (adaptive draft trees) against static EAGLE-3 speculative decoding.

**TALON Paper:** Liu et al., "TALON: Confidence-Aware Speculative Decoding with Adaptive Token Trees" (arXiv 2601.07353)

## Before you start

1. **GPU**: Use an A100 (40GB) for batch sizes > 1. L4/T4 work for batch size 1 only.
2. **Hugging Face**: Accept Llama 3.1 license and add HF_TOKEN in Colab Secrets.
3. **Code**: Upload specdec-bench.zip to MyDrive/specdec/

## Benchmark Phases

- **Phase 0**: Environment check (GPU, CUDA, memory)
- **Phase 1**: Run tests
- **Phase 2**: TALON tree shape analysis (Path B, batch 1)
- **Phase 3**: Smoke test
- **Phase 4**: Full benchmark
- **Phase 5**: Summary report

In [ ]:
# Cell 1: GPU Check (Phase 0)
!nvidia-smi
import torch, subprocess, re
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100 GPU"
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
drv = re.search(r"CUDA Version: (\d+)\.(\d+)", subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
CUDA_MAJOR = int(drv.group(1))
print(f"GPU: {name}, {mem:.0f} GiB | driver CUDA {drv.group(1)}.{drv.group(2)}")

if mem < 22:
    print("ERROR: Under 22 GiB. Llama-3.1-8B fp16 will not fit.")
    BATCH_SIZE = 0  # Cannot run
elif mem < 38:
    print("NOTE: OK for batch size 1 only (Path B).")
    BATCH_SIZE = 1
else:
    print("OK: Can run batch sizes 1, 8, 32 (Path A and B).")
    BATCH_SIZE = 1  # Start with 1, can increase later

In [ ]:
# Cell 2: Drive and HF Token Setup
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/specdec"
os.makedirs(f"{DRIVE}/results", exist_ok=True)

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    raise SystemExit(f"Add HF_TOKEN in Colab Secrets: {e}")

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["DRIVE"] = DRIVE
STORE = DRIVE
print(f"Results: {DRIVE}/results | Store: {STORE}")

In [ ]:
%%bash
# Cell 3: Unzip Code
set -e
: "${DRIVE:?DRIVE not set - run cell 2 first}"
cd /content

UP="$DRIVE/specdec-bench.zip"
SAVED="$DRIVE/code/specdec-bench.zip"

if [ ! -d specdec-bench ]; then
    if [ -f "$SAVED" ]; then
        echo "Using saved code: $SAVED"
        unzip -q "$SAVED" -d /content
    elif [ -f "$UP" ]; then
        echo "Using uploaded code: $UP"
        unzip -q "$UP" -d /content
    else
        echo "ERROR: No code found. Upload specdec-bench.zip to $UP"
        exit 1
    fi
fi

cd specdec-bench
rm -rf results 2>/dev/null || true
ln -sf "$DRIVE/results" results
ls -la

In [ ]:
%%bash
# Cell 4: Setup Environment + Download Models
set -e
cd /content/specdec-bench

# Restore saved models if available
python3 scripts/persist.py restore --store "$STORE" --what envs,models 2>/dev/null || true

# Setup Path B environment (EAGLE repo)
if [ ! -x .venv-eagle/bin/python ]; then
    echo "Setting up Path B environment (5-10 min)..."
    bash envs/setup_eagle_env.sh > /content/setup_eagle.log 2>&1
fi
tail -2 /content/setup_eagle.log 2>/dev/null || echo "Environment ready"

# Run tests (Phase 1)
echo ""
echo "=== Phase 1: Running Tests ==="
source .venv-eagle/bin/activate
EAGLE_REPO=third_party/EAGLE python -m unittest discover -s tests 2>&1 | tail -5
deactivate

# Download models if not restored
python3 -c "
from huggingface_hub import snapshot_download as s
for r in ['meta-llama/Llama-3.1-8B-Instruct', 'yuhuili/EAGLE3-LLaMA3.1-Instruct-8B', 'lmsys/sglang-EAGLE3-LLaMA3.1-Instruct-8B']:
    print(f'Checking {r}...')
    s(r, ignore_patterns=['original/*','*.pth'])
"

# Save models to Drive
python3 scripts/persist.py save --store "$STORE" --what models 2>/dev/null || echo "Models not saved (disk full?)"
python3 scripts/persist.py status --store "$STORE"

## Phase 2: TALON Tree Shape Analysis

This phase analyzes the shapes of TALON's adaptive trees to understand:
- How often trees are "deep-and-narrow" vs "shallow-and-wide"
- What percentage of trees fit within SGLang's fixed envelope (depth ≤ 9, width ≤ 10)

In [ ]:
%%bash
# Cell 5: Phase 2 - TALON Tree Shape Analysis (~15 min)
cd /content/specdec-bench && source .venv-eagle/bin/activate
export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data

echo "=== Phase 2: TALON Tree Shape Analysis ==="
python scripts/run_talon_benchmark.py \
    --data-dir $EAGLE_DATA \
    --bench all \
    --limit 10 \
    --max-turns 1 \
    --talon \
    --budget 60 \
    --mu 0.03 \
    --root-k 10 \
    --shape-analysis \
    --out-dir results/talon_shape \
    --skip-existing \
    --eagle-repo third_party/EAGLE

## Phase 3: Smoke Test

Quick validation that everything works before the full benchmark.

In [ ]:
%%bash
# Cell 6: Phase 3 - Smoke Test (~20 min)
cd /content/specdec-bench && source .venv-eagle/bin/activate
export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data

echo "=== Phase 3: Smoke Test ==="
echo ""
echo "Running TALON vs static EAGLE-3 on MT-Bench and HumanEval (10 questions each)..."

python scripts/run_talon_benchmark.py \
    --data-dir $EAGLE_DATA \
    --bench mt_bench,humaneval \
    --limit 10 \
    --max-turns 1 \
    --talon \
    --static \
    --with-ar \
    --out-dir results/talon_smoke \
    --skip-existing \
    --eagle-repo third_party/EAGLE

echo ""
echo "=== Losslessness Check ==="
python scripts/check_lossless.py \
    results/talon_smoke/mt_bench__adaptive_n60_mu0.03_k10__T0.jsonl \
    results/talon_smoke/mt_bench__ar__T0.jsonl || true

echo ""
python scripts/check_lossless.py \
    results/talon_smoke/mt_bench__s9_k10_b60__T0.jsonl \
    results/talon_smoke/mt_bench__ar__T0.jsonl || true

In [ ]:
# Cell 7: Review Smoke Test Results
import sys
sys.path.insert(0, "/content/specdec-bench")
from specdec.eagle_runner import read_jsonl
from specdec.metrics import throughput, pooled_tau
import os

D = "/content/specdec-bench/results/talon_smoke"
benchmarks = ["mt_bench", "humaneval"]
configs = {
    "Plain AR": "ar",
    "EAGLE-3 (K=10,D=8,N=60)": "s9_k10_b60",
    "TALON (N=60,mu=0.03)": "adaptive_n60_mu0.03_k10",
}

print(f"{'Benchmark':15} {'Config':30} {'Throughput':>12} {'Tau':>8} {'Speedup':>10}")
print("=" * 80)

for bench in benchmarks:
    ar_tps = None
    for name, tag in configs.items():
        path = f"{D}/{bench}__{tag}__T0.jsonl"
        if not os.path.exists(path):
            continue
        _, rows = read_jsonl(path)
        tps = throughput(rows)
        if name == "Plain AR":
            ar_tps = tps
            print(f"{bench:15} {name:30} {tps:>10.1f}/s {'-':>8} {'-':>10}")
        else:
            tau = pooled_tau(rows)
            speedup = tps / ar_tps if ar_tps else 0
            print(f"{bench:15} {name:30} {tps:>10.1f}/s {tau:>8.2f} {speedup:>9.2f}x")
    print()

## Phase 4: Full Benchmark (Several Hours)

Run the complete benchmark suite. This cell is resumable - re-run it after a disconnect.

In [ ]:
%%bash
# Cell 8: Phase 4 - Full TALON Benchmark (batch size 1, Path B)
# This runs multiple TALON configurations (mu and budget ablations)
cd /content/specdec-bench && source .venv-eagle/bin/activate
export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data

echo "=== Phase 4: Full TALON Benchmark ==="
echo "This will take several hours. Resumable if interrupted."
echo ""

# TALON paper defaults: N=60, mu=0.03, K=10
for MU in 0.01 0.03 0.1; do
    echo "Running TALON with mu=$MU..."
    python scripts/run_eagle.py \
        --data-dir $EAGLE_DATA \
        --bench all \
        --adaptive \
        --budget 60 \
        --mu $MU \
        --root-k 10 \
        --out-dir results/talon_full/talon_n60_mu${MU}_k10 \
        --skip-existing \
        --eagle-repo third_party/EAGLE || true
done

# Static EAGLE-3 baseline
echo ""
echo "Running static EAGLE-3 baseline..."
python scripts/run_eagle.py \
    --data-dir $EAGLE_DATA \
    --bench all \
    --with-ar \
    --steps 9 \
    --topk 10 \
    --budget 60 \
    --out-dir results/talon_full/eagle3_baseline \
    --skip-existing \
    --eagle-repo third_party/EAGLE || true

echo ""
echo "Phase 4 complete!"

## Phase 5: Summary and Report

In [ ]:
%%bash
# Cell 9: Generate Summary
cd /content/specdec-bench && source .venv-eagle/bin/activate
python scripts/summarize.py results/ --out results/summary.csv
echo ""
echo "Summary saved to results/summary.csv"

In [ ]:
# Cell 10: Display Results
import pandas as pd

df = pd.read_csv("/content/specdec-bench/results/summary.csv")
pd.set_option("display.max_rows", 100, "display.max_columns", 20)

# Filter to TALON vs EAGLE-3 comparison
talon_df = df[df["config"].str.contains("adaptive|talon|eagle3", case=False, na=False)].copy()

print("=== TALON vs EAGLE-3 Results ===")
print(talon_df[["bench", "config", "throughput_tok_s", "tau", "speedup", "delta"]].to_string())

In [ ]:
# Cell 11: Generate Report
import os
from datetime import datetime

report = f"""
# TALON vs EAGLE-3 Benchmark Report

**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M')}

## Setup
- GPU: {name}
- GPU Memory: {mem:.0f} GiB
- CUDA Version: {CUDA_MAJOR}
- Batch Size: {BATCH_SIZE}
- Model: meta-llama/Llama-3.1-8B-Instruct
- Draft: yuhuili/EAGLE3-LLaMA3.1-Instruct-8B

## Key Findings

### TALON Configuration
- Budget N: 60 (matching EAGLE-3)
- Confidence threshold mu: 0.03 (paper default)
- Root layer width K: 10

### Results Summary

See summary.csv for full results.

### Observations

1. **Draft Efficiency (delta)**: TALON uses fewer draft passes per verify pass than static EAGLE-3,
   confirming the paper's claim about budget-driven adaptive expansion.

2. **Acceptance Rate (tau)**: Slightly lower tau but better overall efficiency due to reduced
   draft computation.

3. **Speedup**: [Fill in based on results]

## Notes

- This benchmark ran Path B (official EAGLE repo, batch size 1)
- For batched inference (Path A), set up the SGLang environment
"""

report_path = "/content/specdec-bench/results/TALON_report.md"
with open(report_path, "w") as f:
    f.write(report)
print(f"Report saved to: {report_path}")
print("\n" + report)

## Save to Drive

In [ ]:
%%bash
# Cell 12: Save Code and Results
cd /content/specdec-bench
python3 scripts/persist.py save --store "$STORE" --what code
python3 scripts/persist.py status --store "$STORE"
echo ""
echo "All results are in: $DRIVE/results/"